# RQ2 — Robustness Evaluation (DBpedia)

Corrected version with the same three fixes applied to the Wikidata notebook:
1. **Qwen baseline** is now `Qwen/Qwen3-Embedding-0.6B` (an actual embedding model), not the `Qwen2.5-0.5B` chat model.
2. **Self-exclusion in MRR**: the anchor's own embedding is excluded from its retrieval pool (`scores[anchor_idx] = -inf`). Previously the self-match capped every rank at ≥2 and distorted MRR.
3. **Explicit L2 normalization** for every encoder.

Also adds a POOL CHECK print so this eval can be cross-checked against the DBpedia RQ1 eval (they must use the same corpus construction).

In [1]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sentence_transformers import SentenceTransformer

# ---- Project path setup (notebook-safe) ----
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from sparqlce.embedder import SPARQLEmbedder   # noqa: E402

print(f"✓ Project root: {_PROJECT_ROOT}")

/upb/users/m/manzoor/profiles/unix/cs/.conda/envs/conda-jupyterhub/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Project root: /upb/users/m/manzoor/profiles/unix/cs/SPARQL-Embeddings-main-refactored


In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ROBUSTNESS_FILE = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ2.json")
TEST_FILE       = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json")
SPARQLCE_CKPT   = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")
OUTPUT_DIR      = os.path.join(_PROJECT_ROOT, "results", "robustness")

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# ==========================================
# 1. Model wrappers — ALL return L2-normalized embeddings
# ==========================================
def _l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norms, 1e-8, None)


class SparqlCEEncoder:
    def __init__(self, ckpt_path, device='cuda'):
        self.device = device
        self.embedder = SPARQLEmbedder(ckpt_path)

    def encode(self, queries, batch_size=32):
        if hasattr(self.embedder, 'embed'):
            emb = self.embedder.embed(queries)
        elif hasattr(self.embedder, 'encode'):
            emb = self.embedder.encode(queries)
        else:
            raise AttributeError("SPARQLEmbedder exposes neither embed() nor encode()")
        return _l2_normalize(emb)


class E5Encoder:
    def __init__(self, model_name='intfloat/e5-base-v2', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, queries):
        prefixed = [f'query: {q}' for q in queries]
        return self.model.encode(prefixed, convert_to_numpy=True,
                                 show_progress_bar=False, normalize_embeddings=True)


class QwenEncoder:
    # FIX 1: use the actual embedding model, not the chat model
    def __init__(self, model_name='Qwen/Qwen3-Embedding-0.6B', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, queries):
        return self.model.encode(queries, convert_to_numpy=True,
                                 show_progress_bar=False, normalize_embeddings=True)


class TFIDFEncoder:
    def __init__(self, corpus):
        self.vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2),
                                          max_features=10000)
        self.vectorizer.fit(corpus)

    def encode(self, queries):
        v = self.vectorizer.transform(queries).toarray()
        return _l2_normalize(v)   # FIX 3: L2-normalize TF-IDF too

In [4]:
class MiniLMEncoder:
    """SBERT MiniLM-L6-v2 encoder with query prefix (optional)."""
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, queries):
        # MiniLM does not require a special prefix; just encode directly
        return self.model.encode(queries, convert_to_numpy=True,
                                 show_progress_bar=False, normalize_embeddings=True)


class MPNetEncoder:
    """SBERT MPNet-base-v2 encoder with query prefix (optional)."""
    def __init__(self, model_name='sentence-transformers/all-mpnet-base-v2', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, queries):
        # MPNet also uses plain sentences; no special prefix needed
        return self.model.encode(queries, convert_to_numpy=True,
                                 show_progress_bar=False, normalize_embeddings=True)

In [5]:
# ==========================================
# 2. Metrics
# ==========================================
def compute_cosine_sim(emb_orig, emb_pert):
    a = torch.tensor(np.asarray(emb_orig, dtype=np.float32))
    b = torch.tensor(np.asarray(emb_pert, dtype=np.float32))
    return F.cosine_similarity(a, b, dim=1).numpy()


def mrr_against_corpus(anchor_emb, corpus_emb, anchor_idx, pos_idx):
    """
    MRR of a single anchor against the full corpus.
    FIX 2: exclude the anchor's own index so self-similarity (=1.0)
    does not occupy rank 1 and distort the score.
    """
    scores = sk_cosine(anchor_emb, corpus_emb).flatten()
    scores[anchor_idx] = -np.inf            # exclude the query itself
    best = scores[pos_idx].max()
    rank = 1 + int(np.sum(scores > best))
    return 1.0 / rank

In [6]:
# ==========================================
# 3. Load data & build corpus
# ==========================================
with open(ROBUSTNESS_FILE, 'r', encoding='utf-8') as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs)} DBpedia robustness pairs")
for i, p in enumerate(pairs):
    p['_idx'] = i

test_data_loaded = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    try:
        test_data_loaded = json.load(f)
    except json.JSONDecodeError:
        f.seek(0)
        for line in f:
            line = line.strip()
            if line:
                try:
                    test_data_loaded.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
print(f"Loaded {len(test_data_loaded)} items from triplet dataset.")

corpus_queries = []
query_to_idx = {}
anchor_pos_indices = defaultdict(list)

def _add(q):
    q = q.strip()
    if q and q not in query_to_idx:
        query_to_idx[q] = len(corpus_queries)
        corpus_queries.append(q)
    return query_to_idx.get(q)

for item in test_data_loaded:
    anchor = item.get('anchor_query', item.get('query', '')).strip()
    if not anchor:
        continue
    _add(anchor)

    positives = item.get('positive_query', item.get('positive_queries', []))
    if isinstance(positives, str):
        positives = [positives]
    for p in positives:
        p_text = p['query'].strip() if isinstance(p, dict) else p.strip()
        idx = _add(p_text)
        anchor_pos_indices[anchor].append(idx)

    negatives = item.get('negative_query', item.get('negative_queries', []))
    if isinstance(negatives, str):
        negatives = [negatives]
    for n in negatives:
        n_text = n['query'].strip() if isinstance(n, dict) else n.strip()
        _add(n_text)

pert_by_anchor = defaultdict(list)
for p in pairs:
    p['original'] = p['original'].strip()
    p['perturbed'] = p['perturbed'].strip()
    pert_by_anchor[p['original']].append(p)

print("\n[POOL CHECK]")
print(f"  corpus size                = {len(corpus_queries)}")
print(f"  anchors with positives     = {len(anchor_pos_indices)}")
print(f"  anchors with perturbations = {len(pert_by_anchor)}")
_evaluable = [a for a in anchor_pos_indices if a in pert_by_anchor]
print(f"  evaluable anchors (both)   = {len(_evaluable)}")

Loaded 7578 DBpedia robustness pairs
Loaded 337 items from triplet dataset.

[POOL CHECK]
  corpus size                = 1677
  anchors with positives     = 337
  anchors with perturbations = 1677
  evaluable anchors (both)   = 337


In [7]:
# ==========================================
# 4. Instantiate models & evaluate
# ==========================================

models = {
     'sparqlce': SparqlCEEncoder(SPARQLCE_CKPT, DEVICE),
     'e5':       E5Encoder(device=DEVICE),
     'qwen':     QwenEncoder(device=DEVICE),
     'tfidf':    TFIDFEncoder(corpus_queries),
     'minilm':   MiniLMEncoder(device=DEVICE),      # added
     'mpnet':    MPNetEncoder(device=DEVICE),       # added
}

results = {}


for name, encoder in models.items():
    print(f"\n=== Evaluating {name} ===")
    originals  = [p['original']  for p in pairs]
    perturbeds = [p['perturbed'] for p in pairs]

    emb_orig   = encoder.encode(originals)
    emb_pert   = encoder.encode(perturbeds)
    cos_sim    = compute_cosine_sim(emb_orig, emb_pert)
    corpus_emb = encoder.encode(corpus_queries)

    mrr_orig_total, mrr_pert_total = [], []
    for anchor, pos_idx in anchor_pos_indices.items():
        if anchor not in pert_by_anchor or len(pos_idx) == 0:
            continue
        anchor_idx = query_to_idx[anchor]
        a_emb = corpus_emb[anchor_idx:anchor_idx + 1]

        mrr_orig_total.append(
            mrr_against_corpus(a_emb, corpus_emb, anchor_idx, pos_idx)
        )

        versions = []
        for pert_pair in pert_by_anchor[anchor]:
            p_emb = emb_pert[pert_pair['_idx']:pert_pair['_idx'] + 1]
            versions.append(
                mrr_against_corpus(p_emb, corpus_emb, anchor_idx, pos_idx)
            )
        if versions:
            mrr_pert_total.append(float(np.mean(versions)))

    mean_orig = float(np.mean(mrr_orig_total)) if mrr_orig_total else 0.0
    mean_pert = float(np.mean(mrr_pert_total)) if mrr_pert_total else 0.0

    by_type = defaultdict(list)
    for p in pairs:
        by_type[p['perturbation_type']].append(p['_idx'])
    per_type = {
        pt: {'mean_cos_sim': float(cos_sim[idxs].mean()), 'n_pairs': len(idxs)}
        for pt, idxs in by_type.items()
    }

    results[name] = {
        'overall_mean_cos_sim': float(cos_sim.mean()),
        'overall_std_cos_sim':  float(cos_sim.std()),
        'mrr_original':  mean_orig,
        'mrr_perturbed': mean_pert,
        'mrr_drop':      mean_orig - mean_pert,
        'n_evaluable_anchors': len(mrr_orig_total),
        'per_type': per_type,
    }
    print(f"  cos_sim={results[name]['overall_mean_cos_sim']:.4f}  "
          f"MRR orig={mean_orig:.4f}  pert={mean_pert:.4f}  "
          f"drop={mean_orig - mean_pert:+.4f}  "
          f"(n={len(mrr_orig_total)} anchors)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8501.82it/s]



=== Evaluating minilm ===
  cos_sim=0.9732  MRR orig=0.0614  pert=0.0635  drop=-0.0021  (n=337 anchors)

=== Evaluating mpnet ===
  cos_sim=0.9769  MRR orig=0.0835  pert=0.0901  drop=-0.0066  (n=337 anchors)


In [8]:
# ==========================================
# 5. Save results & print summary
# ==========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, 'robustness_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print("\n========== DBpedia Robustness Comparison ==========")
print(f"{'Model':<10} {'CosSim':>8} {'MRR Orig':>9} {'MRR Pert':>9} {'MRR Drop':>9}")
for name in models:
    r = results[name]
    print(f"{name:<10} {r['overall_mean_cos_sim']:8.4f} "
          f"{r['mrr_original']:9.4f} {r['mrr_perturbed']:9.4f} {r['mrr_drop']:9.4f}")

print("\nPer perturbation type (mean cosine similarity | n_pairs):")
for name in models:
    print(f"\n{name}:")
    for ptype, stats in results[name]['per_type'].items():
        print(f"  {ptype:<25}: {stats['mean_cos_sim']:.4f}  (n={stats['n_pairs']})")

print("\n----- Headline check -----")
sp = results['sparqlce']['mrr_perturbed']
print(f"SparqlCE perturbed MRR: {sp:.4f}")
for name in models:
    if name == 'sparqlce':
        continue
    bo = results[name]['mrr_original']
    rel = '>' if sp > bo else '≤'
    print(f"  vs {name:8s} original {bo:.4f}: SparqlCE perturbed {rel} by {sp - bo:+.4f}")

print(f"\n✓ Saved to: {out_path}")


========== DBpedia Robustness Comparison ==========
Model        CosSim  MRR Orig  MRR Pert  MRR Drop
minilm       0.9732    0.0614    0.0635   -0.0021
mpnet        0.9769    0.0835    0.0901   -0.0066

Per perturbation type (mean cosine similarity | n_pairs):

minilm:
  variable_renaming        : 0.9864  (n=1677)
  redundant_optional       : 0.9671  (n=1474)
  filter_true              : 0.9719  (n=1677)
  distinct_toggle          : 0.9559  (n=1677)
  triple_shuffle           : 0.9983  (n=873)
  union_duplication        : 0.9282  (n=100)
  subquery_wrapping        : 0.9823  (n=100)

mpnet:
  variable_renaming        : 0.9833  (n=1677)
  redundant_optional       : 0.9764  (n=1474)
  filter_true              : 0.9794  (n=1677)
  distinct_toggle          : 0.9625  (n=1677)
  triple_shuffle           : 0.9947  (n=873)
  union_duplication        : 0.9177  (n=100)
  subquery_wrapping        : 0.9805  (n=100)

----- Headline check -----


KeyError: 'sparqlce'